### Imports

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import re
import urllib3
import csv
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

from langchain.chains import RetrievalQA
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from langchain.schema import Document

### Load WebPage Text

In [ ]:
def load_webpage_text(self, url):
    try:
        resp = requests.get(url, verify=False, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")

        # Get page title from <title> tag or fallback
        title_tag = soup.title.string.strip() if soup.title else "No Title"

        texts = soup.stripped_strings
        full_text = " ".join(texts)

        if len(full_text) < 50:  # Skip pages with too little text
            print(f"Skipped {url} due to short content")
            return []

        doc = Document(
            page_content=full_text,
            metadata={
                "source": url,
                "title": title_tag
            }
        )
        print(f"Loaded webpage '{title_tag}' from {url} (length {len(full_text)})")
        return [doc]
    except Exception as e:
        print(f"Failed to load webpage text from {url} - {e}")
        return []